In [24]:
"""
The next type is Advanced RAG. USING QUERY REWRITE

Unlike Naive RAG, Advanced RAG improves retrieval quality through techniques such as query rewriting, metadata filtering, reranking and better chunking strategies.

For example, if a user asks "Tell me about plant?", a query transformation step could rewrite it as "Explain the characteristics, parts and classification of plants" before sending it to the retriever.

The objective is to retrieve more relevant context before passing it to the LLM.
"""

'\nThe next type is Advanced RAG. USING QUERY REWRITE\n\nUnlike Naive RAG, Advanced RAG improves retrieval quality through techniques such as query rewriting, metadata filtering, reranking and better chunking strategies.\n\nFor example, if a user asks "Tell me about plant?", a query transformation step could rewrite it as "Explain the characteristics, parts and classification of plants" before sending it to the retriever.\n\nThe objective is to retrieve more relevant context before passing it to the LLM.\n'

In [25]:
from langchain_community.document_loaders import TextLoader
# LOAD   LOAD   LOAD   LOAD
loader = TextLoader("./notes/fruits_and_vegetables.txt")

documents = loader.load()

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#   SPLIT   SPLIT   SPLIT   SPLIT
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=1
)

chunks = text_splitter.split_documents(documents)

In [27]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)
vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("./faiss_index")


# Retrieve — load back and search
vectorstore = FAISS.load_local(
    "./faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)
results = vectorstore.similarity_search("Tell us about Apple", k=1)

for doc in results:
    print(doc.page_content)
    print("---")

An apple and a carrot may both be crisp and can both be eaten raw. However, an apple is a botanical fruit because it develops from a flower and contains seeds, while a carrot is a root vegetable.

A tomato and a red pepper may both be red and are commonly used in salads. Botanically, both are fruits because they develop from flowers and contain seeds. In everyday cooking, however, both are usually treated as vegetables.
---


In [28]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Advanced RAG using Query Rewriting --- ReWrite Query before calling LLM
 
llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)

query_rewrite_prompt = ChatPromptTemplate.from_template(
    """
    Rewrite the user's question into a clearer search query
    for retrieving relevant information from documents, 
    in complete sentence. 

    Do not answer the question.
    Do not change its original intent.
    Return only the rewritten query.

    User question: {question}
    """
)
query_rewriter = query_rewrite_prompt | llm | StrOutputParser()


""" 
# Test call is below, uncomment to see the rewritten query.
rewritten_query = query_rewriter.invoke(
    "Which fruit is rich in vitamin C?"
)
print(rewritten_query)
"""


' \n# Test call is below, uncomment to see the rewritten query.\nrewritten_query = query_rewriter.invoke(\n    "Which fruit is rich in vitamin C?"\n)\nprint(rewritten_query)\n'

In [29]:
"""
You're observing correctly, but the distinction is between the retrieval query and the final question. In "context": query_rewriter | retriever, the original user query is first rewritten by Qwen, and that rewritten query is sent to FAISS to retrieve relevant Documents; the retrieved Documents become the context. Meanwhile, "question": RunnablePassthrough() independently preserves the original user query without rewriting it. Finally, your prompt receives both: the retrieved context and the original question. We rewrite the query used for retrieval, not the question presented to the final LLM.

Earlier when we did Naive RAG, while building

the augmented content, we used vectorstore.similarity_search

this was not AI, just inbuilt vector text

HERE in advanced RAG with Query Rewrite,
we are using AI to rewrite the query before sending it to FAISS for retrieval. 

Notice the difference between Naive and Advanced RAG,
We generate the augmented content through the query string
which is enhanced by AI
This is a key difference between Naive RAG and Advanced RAG.

Observe that augment text is passed to LLM under the field context.
"""

'\nYou\'re observing correctly, but the distinction is between the retrieval query and the final question. In "context": query_rewriter | retriever, the original user query is first rewritten by Qwen, and that rewritten query is sent to FAISS to retrieve relevant Documents; the retrieved Documents become the context. Meanwhile, "question": RunnablePassthrough() independently preserves the original user query without rewriting it. Finally, your prompt receives both: the retrieved context and the original question. We rewrite the query used for retrieval, not the question presented to the final LLM.\n\nEarlier when we did Naive RAG, while building\n\nthe augmented content, we used vectorstore.similarity_search\n\nthis was not AI, just inbuilt vector text\n\nHERE in advanced RAG with Query Rewrite,\nwe are using AI to rewrite the query before sending it to FAISS for retrieval. \n\nNotice the difference between Naive and Advanced RAG,\nWe generate the augmented content through the query st

In [30]:
# FAISS stores vectors for searching, while 
# the Retriever fetches the corresponding original text for the LLM.
# IMP:the Retriever doesn't convert vectors back into English; 
# FAISS uses them to locate the original stored Documents, 
# which already contain the English text.

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

#Original prompt for LLM, final one, which has user's query
prompt = ChatPromptTemplate.from_template(
    """
Answer the question based only on the context below.

Context: {context}

Question: {question}
    """
)


def print_rewritten_query(query):
    print("Rewritten Query:", query)
    return query


chain = (
    {
        "context": query_rewriter | print_rewritten_query | retriever ,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke("What is Apple?")
print(response)

Rewritten Query: What is the meaning or significance of Apple?
An apple is a botanical fruit because it develops from a flower and contains seeds, while a carrot is a root vegetable.
